# Therapeutic areas

Each disease ontology term is assigned one therapeutic area: the first therapeutic-area root
among its ontology ancestors. Terms descending from no root become `other`. Methods
"Therapeutic area assignment to studies".

Two orderings of the same 23 roots are carried, because the published analysis used both:

| column | order | used by |
| --- | --- | --- |
| `primaryTherapeuticArea` | Supplementary Table 9, `genetic, familial or congenital disease` third | the gene-level analysis (Results 5) |
| `primaryTherapeuticAreaLegacy` | the pre-refactor qualifying-dataset notebook, that area last but one | the variant and cluster analyses (Results 4) |

Only the published order reproduces the gene-level counts (4,743 genes in more than one area,
mean 2.53, max 21); only the legacy order reproduces the cluster counts. Both are kept so the
manuscript reproduces exactly. Unify on the published order afterwards, see GAPS.md.

Writes `efo_therapeutic_area` and `study_therapeutic_areas`.

In [ ]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

In [ ]:
studies = session.spark.read.parquet(paper.release("study"))
disease = session.spark.read.parquet(paper.release("disease") + "/disease.parquet").select("id", "ancestors")


def first_area(hierarchy):
    """First area of the hierarchy present in a disease's ancestors, else "other"."""
    return f.coalesce(*[f.when(f.array_contains("ancestors", area), f.lit(area)) for area in hierarchy], f.lit("other"))


efo_ta = (
    disease.withColumn("primaryTherapeuticArea", first_area(paper.THERAPEUTIC_AREAS))
    .withColumn("primaryTherapeuticAreaLegacy", first_area(paper.THERAPEUTIC_AREAS_LEGACY))
    .join(studies.select(f.explode("diseaseIds").alias("efo")), f.col("id") == f.col("efo"), "semi")
    .select("id", "primaryTherapeuticArea", "primaryTherapeuticAreaLegacy")
)
efo_ta.write.mode("overwrite").parquet(paper.derived("efo_therapeutic_area"))

efo_ta = session.spark.read.parquet(paper.derived("efo_therapeutic_area")).cache()
print("ontology terms used by studies:", efo_ta.count())
print(
    "terms the two orders disagree on:",
    efo_ta.filter(f.col("primaryTherapeuticArea") != f.col("primaryTherapeuticAreaLegacy")).count(),
)

## GWAS studies annotated with their therapeutic areas

In [ ]:
study_areas = (
    studies.select("studyId", f.explode("diseaseIds").alias("id"))
    .join(efo_ta, on="id", how="inner")
    .groupBy("studyId")
    .agg(
        f.collect_set("primaryTherapeuticArea").alias("mappedTherapeuticAreas"),
        f.collect_set("primaryTherapeuticAreaLegacy").alias("mappedTherapeuticAreasLegacy"),
    )
)

gwas = (
    studies.filter(f.col("studyType") == "gwas")
    .join(study_areas, on="studyId", how="left")
    .withColumn("measurement", f.array_contains("mappedTherapeuticAreas", paper.MEASUREMENT))
    .withColumn("binaryLessCases", f.when(f.col("nCases") < f.col("nControls"), True).otherwise(False))
    .withColumns(
        {
            column: f.when(f.array_contains("mappedTherapeuticAreasLegacy", area), 1).otherwise(0)
            for area, column in paper.TA_COLUMNS.items()
        }
    )
)
gwas = gwas.withColumn("totalTherapeuticAreas", sum(f.col(c) for c in paper.TA_COLUMNS.values()))
gwas.write.mode("overwrite").parquet(paper.derived("study_therapeutic_areas"))
print("GWAS studies:", session.spark.read.parquet(paper.derived("study_therapeutic_areas")).count())

## Cross-check against the pre-refactor table

In [ ]:
new = session.spark.read.parquet(paper.derived("study_therapeutic_areas"))
ref = session.spark.read.parquet(paper.baseline("gwas_w_therapeutic_areas"))
columns = ["studyId", "measurement", "binaryLessCases", "totalTherapeuticAreas", *paper.TA_COLUMNS.values()]
print("rows:", new.count(), ref.count())
print("rows that differ:", new.select(columns).subtract(ref.select(columns)).count())